# Preprocesamiento y Modelado — Telco Customer Churn

## Contenido
1. Limpieza de datos
2. División train/test
3. Función reutilizable con Pipeline
4. Entrenamiento de modelos
5. Ranking comparativo
6. Evaluación de modelos
7. Guardado del modelo final

## 1. Limpieza de datos

En el EDA identificamos que `TotalCharges` viene como string en el dataset original de Kaggle y los clientes con `tenure = 0` tienen `TotalCharges` vacío. Se corrige aquí antes de construir el Pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

pd.options.display.float_format = '{:,.4f}'.format

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')

# Convertir TotalCharges a numérico (viene como string en el dataset original)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Imputar NaN en TotalCharges con 0 (clientes con tenure=0 no tienen cargos)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convertir variable objetivo a binario
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Eliminar columna de identificación
df = df.drop(columns=['customerID'], errors='ignore')

print(f'Filas después de limpieza: {df.shape[0]}')
print(f'NaN en TotalCharges: {df["TotalCharges"].isnull().sum()}')
print(f'Distribución Churn: {df["Churn"].value_counts().to_dict()}')

> **Análisis:** Describir el estado del dataset tras la limpieza.

## 2. División train/test

Se dividen los datos en 80% entrenamiento y 20% prueba de forma **estratificada** para preservar la distribución del target en ambos conjuntos.

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Churn en train: {y_train.mean():.3f} | Churn en test: {y_test.mean():.3f}')

> **Análisis:** Verificar que la proporción de churn se mantiene similar en train y test (estratificación correcta).

## 3. Función reutilizable con Pipeline

Se define un `ColumnTransformer` que aplica:
- `OneHotEncoder` a las variables categóricas
- `StandardScaler` a las variables numéricas

Todo dentro de un `Pipeline` para evitar data leakage: las transformaciones se aprenden **solo** sobre `X_train` y se aplican a `X_test`.

In [ ]:
# Identificar tipos de columnas
categoricas = X.select_dtypes(include='object').columns.tolist()
numericas   = X.select_dtypes(include='number').columns.tolist()

print(f'Numéricas   ({len(numericas)}): {numericas}')
print(f'Categóricas ({len(categoricas)}): {categoricas}')

In [ ]:
def train_pipeline(X_train, y_train, model, param_grid, cv=5):
    """Construye un Pipeline con preprocesamiento y ajusta hiperparámetros con GridSearchCV."""
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoricas),
        ('num', StandardScaler(), numericas)
    ])

    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf',  model)
    ])

    cv_estratificado = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv_estratificado,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)
    return grid

> **Análisis:** Explicar por qué se usa Pipeline en lugar de transformar el dataset directamente, y qué rol juega `StratifiedKFold` en la validación cruzada.

## 4. Entrenamiento de modelos

Se entrenan cuatro clasificadores: **Random Forest**, **XGBoost**, **CatBoost** y **LightGBM**. Para cada uno se define un grid de hiperparámetros y se selecciona la combinación que maximiza el AUC-ROC en validación cruzada estratificada de 5 folds.

In [ ]:
modelos = {
    'RandomForest': (
        RandomForestClassifier(random_state=42),
        {
            'clf__n_estimators': [100, 200],
            'clf__max_depth':    [None, 5, 10],
            'clf__max_features': ['sqrt', 'log2'],
            'clf__min_samples_split': [2, 5]
        }
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
        {
            'clf__n_estimators':  [100, 200],
            'clf__max_depth':     [3, 5],
            'clf__learning_rate': [0.05, 0.1],
            'clf__subsample':     [0.8, 1.0]
        }
    ),
    'CatBoost': (
        CatBoostClassifier(random_seed=42, verbose=0),
        {
            'clf__iterations':    [100, 200],
            'clf__depth':         [4, 6],
            'clf__learning_rate': [0.05, 0.1],
            'clf__l2_leaf_reg':   [1, 3]
        }
    ),
    'LightGBM': (
        LGBMClassifier(random_state=42, verbose=-1),
        {
            'clf__n_estimators':  [100, 200],
            'clf__num_leaves':    [31, 63],
            'clf__learning_rate': [0.05, 0.1],
            'clf__subsample':     [0.8, 1.0]
        }
    )
}

In [ ]:
grids = {}
resultados = {}

for nombre, (modelo, params) in modelos.items():
    print(f'Entrenando {nombre}...', end=' ')
    grid = train_pipeline(X_train, y_train, modelo, params)

    y_pred      = grid.predict(X_test)
    y_proba     = grid.predict_proba(X_test)[:, 1]

    auc       = roc_auc_score(y_test, y_proba)
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)

    resultados[nombre] = {
        'AUC':       round(auc,       3),
        'Accuracy':  round(acc,       3),
        'Precision': round(precision, 3),
        'Recall':    round(recall,    3),
        'F1':        round(f1,        3)
    }
    grids[nombre] = grid
    print(f'AUC: {auc:.3f} | Accuracy: {acc:.3f} | F1: {f1:.3f} | Best params: {grid.best_params_}')

> **Análisis:** Describir el rendimiento de cada modelo, qué hiperparámetros resultaron óptimos y si hay diferencias significativas entre ellos.

## 5. Ranking comparativo

In [ ]:
ranking = pd.DataFrame(resultados).T.sort_values('AUC', ascending=False)
display(ranking)

In [ ]:
metricas = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(ranking))
width = 0.15
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'goldenrod']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (metrica, color) in enumerate(zip(metricas, colors)):
    ax.bar(x + i * width, ranking[metrica], width, label=metrica, color=color, edgecolor='black')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(ranking.index, rotation=0)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparación de modelos — todas las métricas')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ranking['AUC'].plot(kind='bar', figsize=(8, 4), color='steelblue', edgecolor='black')
plt.title('Comparación de modelos por AUC')
plt.ylabel('AUC')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Análisis:** Comparar los modelos en todas las métricas. Justificar cuál es la métrica principal para este problema de churn (AUC-ROC vs F1 vs Recall) y por qué.

## 6. Evaluación de modelos

### 6.1 Matrices de confusión

Las matrices de confusión muestran el desempeño de cada modelo sobre el conjunto de prueba.

In [ ]:
fig, axes = plt.subplots(1, len(grids), figsize=(22, 4))

for i, (nombre, grid) in enumerate(grids.items()):
    y_pred = grid.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(
        ax=axes[i], colorbar=False, cmap='Blues'
    )
    axes[i].set_title(nombre)

plt.suptitle('Matrices de confusión', fontsize=13)
plt.tight_layout()
plt.show()

> **Análisis:** Comparar los falsos negativos (predice No Churn cuando es Churn) y falsos positivos de cada modelo. En churn, los falsos negativos son los más costosos: el cliente se va sin que la empresa pueda retenerlo.

### 6.2 Classification report

In [ ]:
for nombre, grid in grids.items():
    print(f'\n{'='*50}')
    print(f'  {nombre}')
    print(f'{'='*50}')
    print(classification_report(y_test, grid.predict(X_test), target_names=['No Churn', 'Churn']))

> **Análisis:** Describir precision, recall y F1 por clase para cada modelo. ¿Cuál tiene mejor recall en la clase Churn?

### 6.3 Curvas ROC

La curva ROC muestra la capacidad de cada modelo para discriminar entre clientes que hacen churn y los que no.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for nombre, grid in grids.items():
    RocCurveDisplay.from_estimator(grid, X_test, y_test, ax=ax, name=nombre)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_title('Curva ROC — Comparación de modelos')
plt.tight_layout()
plt.show()

> **Análisis:** Describir qué modelos tienen mayor área bajo la curva ROC y cómo se separan del clasificador aleatorio (línea diagonal).

### 6.4 Importancia de características

Se extraen las 10 variables más importantes de cada modelo para entender qué factores impulsan el churn.

In [ ]:
def get_feature_names(grid):
    """Extrae nombres de features tras OneHotEncoding desde el Pipeline."""
    prep = grid.best_estimator_.named_steps['prep']
    cat_names = prep.named_transformers_['cat'].get_feature_names_out(categoricas).tolist()
    return cat_names + numericas


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (nombre, grid) in enumerate(grids.items()):
    clf = grid.best_estimator_.named_steps['clf']

    if hasattr(clf, 'feature_importances_'):
        feat_names = get_feature_names(grid)
        importancias = pd.Series(clf.feature_importances_, index=feat_names)
        top10 = importancias.nlargest(10).sort_values()

        top10.plot(kind='barh', ax=axes[i], color='steelblue', edgecolor='black')
        axes[i].set_title(f'Top 10 features — {nombre}')
        axes[i].set_xlabel('Importancia')
    else:
        axes[i].text(0.5, 0.5, f'{nombre}\nno tiene feature_importances_',
                     ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(nombre)

plt.suptitle('Importancia de características por modelo (Top 10)', fontsize=13)
plt.tight_layout()
plt.show()

> **Análisis:** Identificar qué variables aparecen consistentemente entre las más importantes en los cuatro modelos. ¿Coincide con lo observado en el EDA?

## 7. Guardado del modelo final

Se selecciona el modelo con mayor AUC-ROC como modelo final y se guarda en `app/model.joblib` para ser usado por la API FastAPI.

In [ ]:
# Seleccionar el mejor modelo por AUC
mejor_nombre = ranking['AUC'].idxmax()
mejor_grid   = grids[mejor_nombre]

print(f'Modelo seleccionado: {mejor_nombre}')
print(f'AUC en test:         {resultados[mejor_nombre]["AUC"]}')
print(f'Mejores hiperparámetros: {mejor_grid.best_params_}')
print(f'\nPipeline a guardar:')
print(mejor_grid.best_estimator_)

In [ ]:
ruta_modelo = os.path.join('..', 'app', 'model.joblib')
os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)

joblib.dump(mejor_grid.best_estimator_, ruta_modelo)
print(f'✅ Modelo guardado exitosamente en "{ruta_modelo}"')

In [ ]:
# Guardar X_train y X_test para el notebook de interpretabilidad (LIME)
os.makedirs('../data', exist_ok=True)

X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv',   index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv',   index=False)

print('✅ Archivos guardados:')
print(f'   - data/X_train.csv ({X_train.shape[0]} filas, {X_train.shape[1]} columnas)')
print(f'   - data/X_test.csv  ({X_test.shape[0]} filas, {X_test.shape[1]} columnas)')

## Conclusiones del modelado

> Completar con los hallazgos principales:
> - Modelo con mejor desempeño y justificación de la elección
> - Comparación entre modelos (diferencias en AUC, F1, Recall)
> - Variables más importantes para predecir el churn
> - Observaciones sobre falsos negativos vs falsos positivos en el contexto del negocio
> - Posibles mejoras: SMOTE para desbalance, umbral de clasificación ajustado, más hiperparámetros